# Strong Allee effect and harvesting

Assembled from five task fragments. Read the scope/status in each section. Synthetic examples are not empirical observations. The final submission requires completed derivations, algorithms, results, interpretation and AI/data acknowledgements.

In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP01.</strong> Units and assumptions; logistic baseline; nondimensionalisation; A equilibria and stability, phase lines and branch plot.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP01: task not implemented yet; see docs/tasks/")

## 2. Model B, comparison and harvesting history

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP02.</strong> B roots/stability; fair A/B comparisons; up/down ramp with state carry-over; explicit restocking; no invented automatic recovery.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP02: task not implemented yet; see docs/tasks/")

## 3. Collapse thresholds, finite-time extinction and basins
<div style="border-left:4px solid #16803d;padding:10px"><strong>WP03 analysis implemented.</strong> Synthetic deterministic models; final group integration, human reviews and Devam's detailed learning review remain pending.</div>

For a well-mixed population, assume constant \(r>0\), \(0<A<K\), nonnegative harvest, no delays/age structure or other species. Put \(x=N/K,\ \tau=rt,\ a=A/K,\ q=H/(rK),\ e=h/r\). Then
\[
x'=G(x)-q\quad(A),\qquad x'=G(x)-ex\quad(B),\qquad
G(x)=-x^3/a+(1+a)x^2/a-x.
\]
**Threshold derivation.** \(G'=(-3x^2+2(1+a)x-a)/a\). The larger stationary point is
\(x_*=(1+a+\sqrt{1-a+a^2})/3\), in \((a,1)\), with \(G''(x_*)<0\);
the smaller point lies below a. Thus \(q_c=G(x_*)\) and
\(N_*=(K+A+\sqrt{K^2-KA+A^2})/3,\ H_c=rKq_c\).
For B, maximise \(P(x)=G(x)/x=(1-x)(x/a-1)\):
\(P'=(1+a-2x)/a,\ P''=-2/a\), so
\(e_c=(1-a)^2/(4a)\), \(h_c=re_c\).
The logistic class maximum is \(1/4\), giving \(H_{c,\mathrm{logistic}}=rK/4\).
H and h have different units; threshold comparisons also depend on production calibration.

**Algorithm:** compute closed-form thresholds; check them against independent bounded numerical maxima of expanded polynomials; classify basins before any extinction integration.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))

### Basins and critical cases
For \(0<q<q_c\), A has two positive roots \(x_-<x_+\) and signs \(-,+,-\): below the lower root extinction, above it approach to the upper root. Exact roots remain equilibria. For B,
\[
x_\pm=(1+a\pm\sqrt{(1-a)^2-4ae})/2.
\]
At positive roots \(f'_B=xP'\), so the lower is unstable and upper stable; zero has derivative \(-(1+e)<0\).

| Control | A | B |
|---|---|---|
| Zero | Below a approaches zero asymptotically; above a tends to 1 | Same positive-state basins |
| Below fold | Below lower root reaches zero; above tends to upper root | Below lower root approaches zero; above tends to upper root |
| At fold | Below double root reaches zero; above approaches it | Below double root approaches zero; above approaches it |
| Above fold | Every finite positive start reaches zero | Every positive start approaches zero |

Exact initial equilibria stay there in all rows. At a fold the derivative is zero but signs on both sides are negative: **semistable**, attracting only from above.
The classifier explicitly recognises returned roots and marks non-exact starts within \(2\,10^{-12}\) of an unstable/semistable boundary unresolved. It does not merge adjacent harvest controls into the fold or decide survival from a finite horizon.

### Extinction integral and solver checks
On a reaching-zero A path,
\[
\tau_{\rm ext}=\int_0^{x_0}\frac{du}{q-G(u)},\qquad
t_{\rm ext}=\tau_{\rm ext}/r=\int_0^{N_0}\frac{dN}{H-g(N)}.
\]
For a subcritical start below the barrier the denominator is continuous and positive over the whole path; at/above an intervening equilibrium the extinction integral is inapplicable. For \(q>q_c\), \(G\le q_c\) gives \(\tau_{\rm ext}\le x_0/(q-q_c)\).
For zero-yield A and B, smoothness/uniqueness exclude a positive trajectory reaching zero in finite time.

**Algorithm:** classify -> integrate only reaching-zero paths -> compare quadrature with two RK45 tolerances and independently expanded-polynomial DOP853 -> retain every case/status. No equilibrium pole is crossed. Blank times mean no finite extinction or unresolved boundary, not zero.

RK45 uses a shared-stage 5(4) pair for adaptive error control. We take the fifth-order state, locate the first downward zero event for positive-yield A, and include its endpoint without adding a zero tail. Output times are separate from adaptive internal steps. Quadrature error estimates and local RK tolerances are not rigorous global error bounds. References: [SciPy solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [quad](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html).

**Difficult endpoint handling:** near the subcritical barrier, a logarithmic variable removes the vanishing linear factor. At the computed fold an elementary partial-fraction expression evaluates the extinction integral (`fold_closed_form`, no quadrature-error estimate). This avoids missed endpoint contributions for close-below-fold starts. Root rounding is separate from integration error. See `docs/WP03_EXPLAINED.md` for a basics-first explanation and the derivation.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json"),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json"),
]
for label, script, configuration in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))

### Interpret the synthetic curves
The default \(r=0.7,K=100,A=20\) gives \(N_*=70.55050\), \(H_c=36.75966\), \(h_c=0.56\); logistic gives \(17.5\).
The 427 extinction evaluations retain non-finite outcomes rather than turning them into zero. Initial zero is omitted on the logarithmic time axis. Solid basin boundaries mark critical initial populations; the finite colour grid does not replace exact equilibrium classification.

At fixed reaching-zero initial population, increasing catch reduces extinction time. Approaching a basin boundary can make that time very large. A finite simulation horizon without an event does not settle whether extinction happens later. The six event checks include the fold from below and supercritical passage; the observed differences are much smaller than the declared \(2\,10^{-7}+5\,10^{-7}\tau_{\rm ext}\) acceptance tolerance.

### Why collapse can look delayed: the bottleneck
With \(q=q_c+\delta,\ y=x-x_*\), the exact cubic expansion is
\[
q-G(x)=\delta+\beta y^2+y^3/a,\qquad \beta=\sqrt{1-a+a^2}/a.
\]
For a fixed start above \(x_*\), the trajectory crosses a slow region. Integrating the leading denominator gives
\[
\int\frac{dy}{\delta+\beta y^2}
=\frac{\arctan(y\sqrt{\beta/\delta})}{\sqrt{\beta\delta}},
\quad \tau_{\rm ext}\sim\pi/\sqrt{\beta\delta}.
\]
The full near-fold integral is evaluated with \(y=\sqrt{\delta/\beta}\tan\theta\) to resolve its narrow peak.
This is passage after the equilibria disappear, distinct from relaxation to a stable equilibrium. A start already below the bottleneck need not have this divergence. At the fold, approach from above is algebraic, \(y\simeq y_0/(1+\beta y_0\tau)\).

In [ ]:
rows = read_rows("wp03_bottleneck.csv")
for row in rows:
    print("delta/qc =", row["fraction_above_fold"], "| tau*sqrt(delta) =", row["scaled_time"])
print("Leading scaled-time limit:", rows[0]["asymptotic_scaled_time"])
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))

### Optional results: sensitivity and sustainable yield
Holding other dimensional r,K,A fixed, \(g_N(N_*)=0\) eliminates the derivative of \(N_*\) from the maximum:
\[
H_{c,r}=H_c/r,\quad H_{c,K}=rN_*^2(N_*/A-1)/K^2,\quad
H_{c,A}=-rN_*^2(1-N_*/K)/A^2.
\]
Also
\[
h_{c,r}=h_c/r,\quad h_{c,K}=\tfrac r4(1/A-A/K^2),\quad
h_{c,A}=\tfrac r4(-K/A^2+1/K).
\]
Central differences check these derivatives; elasticities are \((p/C)C_p\). An a sweep at fixed r,K is a different experiment.

A's stable-branch yield supremum \(H_c\) lies at the semistable cliff, with no control margin.
For B, \(Y(h)=hN_+(h)=g(N_+(h))\). The production maximiser is on the upper branch, so
\(h_{\rm MSY}=g(N_*)/N_*<h_c,\ Y_{\rm MSY}=H_c\).
Independent bounded numerical maximisation checks the optimum. The example effort margin at MSY is about 6.96%; the initial state must still exceed the unstable root. Separate illustrative A/B policies are not a universal risk ranking.

In [ ]:
summary = json.loads((outputs["analysis"] / "wp03_summary.json").read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(outputs["analysis"] / "wp03_sensitivity_yield.png")))
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))

### Handoff, reproduction and limitations
The early handoff retains six clean A/B CSVs (606 points). WP05 owns noise, frozen splits, inverse PINN/ODE fitting and held-out validation; WP04 supplies the shared neural engine. Keep known conditions/fixed external scales separate from evaluation truth, and never fit against held-out clean values or hidden unknown parameters.

Exact commands and test/evidence mapping: **docs/WP03_ANALYSIS.md**. The experiment regenerates tables and PNG/EPS; manifests record versions, units, tolerances and source/file hashes. All data are synthetic; no empirical species-level conclusion follows.

The report, notebook and EPS files required by the course must ultimately be integrated with all five tasks. **scripts/check_submission_ready.py** aggregates missing final artifacts, unfinished sections and metadata; a successful WP03 run does not make the final group submission ready. Reviews and non-author reproduction remain required.

AI assistance: OpenAI ChatGPT/Codex for this implementation, mathematical writing, tests and documentation, recorded in docs/AI_USAGE.md. Devam's detailed understanding review remains pending as requested.

## 4. Forward PINNs

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP04.</strong> Differentiable network and residual; initial condition/physics loss; optimiser/collocation algorithm; both models and independent solver errors/runtime.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP04: task not implemented yet; see docs/tasks/")

# 5. Inverse PINNs, Identifiability, and Held-Out Validation

**Owner:** @rohanagrawal28 (WP05)

In this section, we estimate unknown parameters of strong-Allee harvesting models from discrete, noisy population observations.
We investigate two central questions:
1. **Structural Identifiability:** Are model parameters uniquely determined by ideal observation trajectories?
2. **Inference & Generalization:** How accurately can parameters and held-out future states be predicted under measurement noise using conventional ODE least squares versus Physics-Informed Neural Networks (PINNs)?

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Automatic Colab & Local path detection
if 'google.colab' in sys.modules:
    for candidate in [Path.cwd(), Path('/content'), Path('/content/Rohan_RK45_Data_and_Code'), Path('/content/mm-allee-harvesting')]:
        if (candidate / 'src/mm_allee').exists():
            if str(candidate / 'src') not in sys.path:
                sys.path.insert(0, str(candidate / 'src'))
            os.chdir(candidate)
            break
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'src/mm_allee').exists()), None)
    if ROOT is not None and str(ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(ROOT / 'src'))

from mm_allee.models import Parameters, dimensional_rhs
from mm_allee.solver import solve_trajectory
from mm_allee.data import generate_noisy_observations, validate_splits, validate_time_windows
from mm_allee.inference import fit_ode_least_squares
from mm_allee.metrics import errors
from mm_allee.validation import compute_parameter_metrics

print('Loaded shared modules successfully.')

## 5.1 Model B Structural Nonidentifiability and Counterexample

Under Model B, harvesting is proportional to population density ($h N$):
$$\frac{dN}{dt} = r N \left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right) - h N$$

Expanding the cubic polynomial:
$$\frac{dN}{dt} = -(r + h) N + r\left(\frac{1}{A} + \frac{1}{K}\right) N^2 - \frac{r}{AK} N^3$$

Any trajectory observed at a single constant effort $h$ reveals at most **three** composite polynomial combinations:
$$c_1 = -(r + h), \quad c_2 = r\left(\frac{1}{A} + \frac{1}{K}\right) = \frac{r(A+K)}{AK}, \quad c_3 = -\frac{r}{AK}$$

Because there are 4 unknown physical parameters $(r, A, K, h)$ but only 3 observable coefficients, the system is **structurally nonidentifiable** from data at a single fixed effort.

In [ ]:
# Demonstrate the exact two-parameter-set counterexample:
p1 = Parameters(r=1.0, A=0.2, K=1.0, h=0.5)
p2 = Parameters(r=1.35, A=0.3, K=0.9, h=0.15)

c1_1, c2_1, c3_1 = -(p1.r + p1.h), p1.r * (1.0 / p1.A + 1.0 / p1.K), -p1.r / (p1.A * p1.K)
c1_2, c2_2, c3_2 = -(p2.r + p2.h), p2.r * (1.0 / p2.A + 1.0 / p2.K), -p2.r / (p2.A * p2.K)

print(f'Parameter Set 1: c1={c1_1:.2f}, c2={c2_1:.2f}, c3={c3_1:.2f}')
print(f'Parameter Set 2: c1={c1_2:.2f}, c2={c2_2:.2f}, c3={c3_2:.2f}')
assert np.isclose(c1_1, c1_2) and np.isclose(c2_1, c2_2) and np.isclose(c3_1, c3_2)

# Solve trajectories from three distinct initial states for both parameter sets
d1, d2 = p1.dimensionless(), p2.dimensionless()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for n0, color in zip([0.4, 0.7, 1.1], colors, strict=True):
    traj1 = solve_trajectory('B', a=d1['a'], harvest=d1['e'], x0=n0 / p1.K, tau_end=p1.r * 4.0, n_points=81)
    traj2 = solve_trajectory('B', a=d2['a'], harvest=d2['e'], x0=n0 / p2.K, tau_end=p2.r * 4.0, n_points=81)
    t1, y1 = traj1.tau / p1.r, traj1.x * p1.K
    t2, y2 = traj2.tau / p2.r, traj2.x * p2.K
    ax.plot(t1, y1, label=f'Set 1 (N0={n0})', color=color, lw=2)
    ax.plot(t2, y2, '--', label=f'Set 2 (N0={n0})', color='black', alpha=0.7)

ax.set_title('Model B Identifiability Counterexample: Identical Trajectories from Different Parameters')
ax.set_xlabel('Time t')
ax.set_ylabel('Population N(t)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5.2 Observations, Noise Generation, and Frozen 60/20/20 Splits

We construct noisy observations according to the contract:
$$N_{\text{obs}}(t_i) = N_{\text{clean}}(t_i) + \sigma \cdot N_{\text{ref}} \cdot \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$
with fixed external scale $N_{\text{ref}} = 100.0$. Observations are never silently clipped.

Splits are strictly partitioned into contiguous time windows:
- **Train (60%):** used exclusively for fitting and data loss.
- **Validation (20%):** used for early stopping and model selection.
- **Held-out Test (20%):** completely frozen until final post-training evaluation.

In [ ]:
p_true = Parameters(r=0.7, K=100.0, A=20.0, H=10.0)
d_true = p_true.dimensionless()
ref_traj = solve_trajectory('A', a=d_true['a'], harvest=d_true['q'], x0=45.0 / 100.0, tau_end=0.7 * 8.0, n_points=101)
times = ref_traj.tau / 0.7
clean_pops = ref_traj.x * 100.0

obs_5pct = generate_noisy_observations(
    times, clean_pops, trajectory_id='a_regular', model='A', noise_sigma=0.05, noise_seed=11, n_ref=100.0
)

idx_tr, idx_va, idx_te = obs_5pct['train_indices'], obs_5pct['val_indices'], obs_5pct['test_indices']
print(f'Train points: {len(idx_tr)} (0 to {times[idx_tr[-1]]:.2f})')
print(f'Val points:   {len(idx_va)} ({times[idx_va[0]]:.2f} to {times[idx_va[-1]]:.2f})')
print(f'Test points:  {len(idx_te)} ({times[idx_te[0]]:.2f} to {times[idx_te[-1]]:.2f})')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(0, times[idx_tr[-1]], color='blue', alpha=0.1, label='Train Split (60%)')
ax.axvspan(times[idx_tr[-1]], times[idx_va[-1]], color='green', alpha=0.1, label='Val Split (20%)')
ax.axvspan(times[idx_va[-1]], times[-1], color='red', alpha=0.1, label='Held-out Test (20%)')
ax.plot(times, clean_pops, 'k-', lw=2, label='Clean Reference')
ax.scatter(times, obs_5pct['observed_population'], color='darkblue', s=20, alpha=0.6, label='Noisy Observations (5% noise)')
ax.set_title('Model A: Observation Splits and Noise Structure')
ax.set_xlabel('Time t')
ax.set_ylabel('Population N(t)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.3 Conventional Nonlinear Least-Squares ODE Baseline

We fit unknown parameters using SciPy's Trust Region Reflective solver on training observations only, with known $K=100.0$ and bounds $r > 0, 0 < A < K, H \ge 0$.

In [ ]:
fit_res = fit_ode_least_squares(
    'A',
    observations=obs_5pct,
    known_conditions={'known_K': 100.0, 'known_initial_population': 45.0},
    fit_params=['r', 'A', 'H'],
    initial_guess={'r': 1.0, 'A': 30.0, 'H': 5.0},
    n_ref=100.0,
)

print(f'ODE Fit Status: {fit_res.status} in {fit_res.wall_seconds:.3f}s')
print('Estimated Parameters:')
for p in ['r', 'A', 'H']:
    true_v = getattr(p_true, p)
    est_v = fit_res.estimated_parameters[p]
    rel_err = abs(est_v - true_v) / true_v * 100
    print(f'  {p}: True = {true_v:.2f}, Estimated = {est_v:.4f} (Error: {rel_err:.2f}%)')

print('Held-out Test Performance:')
print(f'  RMSE: {fit_res.test_metrics["rmse"]:.4f}, Relative L2: {fit_res.test_metrics["relative_l2"]:.4f}')

## 5.4 Inverse PINN with Differentiable Constraints

We optimize network weights and physical parameters jointly, preserving gradients via:
$$r = \mathrm{softplus}(\tilde{r}) > 0, \quad A = K \cdot \sigma(\tilde{A}) \in (0, K), \quad H = \mathrm{softplus}(\tilde{H}) \ge 0$$
Training loss is evaluated **strictly on training observations** with validation early stopping.

In [ ]:
try:
    import torch
    from mm_allee.pinn.network import PINN
    from mm_allee.pinn.inverse import TrainableParameters, InversePINNTrainer

    net = PINN(hidden_layers=[32, 32], activation='tanh', t_ref=1.0, n_ref=100.0)
    tp = TrainableParameters('A', fit_params=['r', 'A', 'H'], known_conditions={'known_K': 100.0})
    trainer = InversePINNTrainer(net, tp, lr_net=1e-3, lr_params=1e-2, n_ref=100.0)
    pinn_res = trainer.fit(obs_5pct, known_conditions={'known_initial_population': 45.0}, epochs=200, patience=30, seed=11)

    print(f'PINN Completed: {pinn_res.status} in {pinn_res.wall_seconds:.2f}s (Final Epoch: {pinn_res.final_epoch})')
    for p in ['r', 'A', 'H']:
        true_v = getattr(p_true, p)
        est_v = pinn_res.estimated_parameters[p]
        rel_err = abs(est_v - true_v) / true_v * 100
        print(f'  {p}: True = {true_v:.2f}, PINN Est = {est_v:.4f} (Error: {rel_err:.2f}%)')
except ImportError:
    print('PyTorch is not installed in this environment; skipping PINN training. Full code is implemented in mm_allee.pinn.')

## 5.5 Key Scientific Takeaways

1. **Parameter Error $\ne$ Trajectory Error:** PINNs can fit trajectory data closely while settling on incorrect parameter estimates due to overparameterization.
2. **Classical ODE Baselines are Strong:** For scalar ODEs with low-dimensional parameter spaces, SciPy nonlinear least squares is faster and achieves superior parameter accuracy.
3. **Structural Identifiability Precedes Optimization:** Model B with unknown effort $h$ is mathematically ambiguous; no optimization algorithm can break this ambiguity without additional independent information (e.g. known $K$ or multiple effort levels).